In [1]:
import pandas as pd
import os
import math
import numpy as np
import json
import random
import re
import copy
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from tqdm import tqdm
import datetime

In [2]:
os.chdir('/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis')

In [3]:
%pwd

'/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis'

In [5]:
nace_description_path = "data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv"
nace_descriptions = pd.read_csv(nace_description_path, sep="\t")

In [6]:
def get_zero_shot_user_prompt(path): 
    with open(path, "r") as f: 
        prompt_json = json.load(f)

    return prompt_json["user_prompt_zero_shot"]

def get_few_shot_user_prompt(path): 
    with open(path, "r") as f: 
        prompt_json = json.load(f)

    return prompt_json["user_prompt_few_shot"]

def get_system_prompt(path): 
    with open(path, "r") as f: 
        prompt_json = json.load(f)

    return prompt_json["system_prompt"]

In [7]:
user_prompt_topic = """
TASK
For the following business sector, create short descriptions of realistic business models for exisiting companies. 

DEFINITION
{includes} {includes_also}

{excludes}

INSTRUCTION
- Create a list of {num_samples} different realistic explanations of a business model 
- The length should be one sentence
- Pick one or multiple subsections for this business model
- Do not explain what you did or what you used
"""

In [8]:
def generate_topic(
        num_samples: int, 
        includes: str,
        includes_also: str, 
        excludes: str,
        prompt_path: str,  
        temperature: float = 0.4, 
        model: str = "gpt-4o-mini"
): 

    # Initialize LLM
    if model == "gpt-4o-mini":
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature
        )
    elif model == "openai/gpt-oss-120b":
        llm = ChatOllama(
            model=model,
            temperature=temperature, 
            base_url="http://10.80.20.127:11434/"
        )


    prompt = ChatPromptTemplate.from_messages([
        ("system", ""),
        ("human", user_prompt_topic)
    ])

    # Adapt excludes
    if excludes != "": 
        excludes = "Excludes: " + excludes

    # Chain
    chain = prompt | llm

    # inputs
    input = {
        "num_samples": num_samples,
        "includes": includes,
        "includes_also": includes_also,
        "excludes": excludes,
        }

    formatted_prompt = prompt.invoke(input)

    # print("Formatted User Prompt:", formatted_prompt.messages[1].content)

    # Run
    response = chain.invoke(input)

    # print(response.content)

    return formatted_prompt, response.content

In [9]:
def generate_synthetic_data(
        num_samples: int, 
        gold_standard: list, 
        includes: str,
        includes_also: str, 
        excludes: str,
        subsections: list,
        prompt_path: str,  
        temperature: float = 0.4, 
        model: str = "gpt-4o-mini", 
        topic: str = None
): 

    # Initialize LLM
    if model == "gpt-4o-mini":
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature
        )
    elif model == "openai/gpt-oss-120b":
        llm = ChatOllama(
            model=model,
            temperature=temperature, 
            base_url="http://10.80.20.127:11434/"
        )

    # Prompt
    if gold_standard == []: 
        #print("Zero Shot!")
        prompt = ChatPromptTemplate.from_messages([
            ("system", get_system_prompt(prompt_path)),
            ("human", get_zero_shot_user_prompt(prompt_path))
        ])
        gold_standard_str = ""
        
    else: 
        prompt = ChatPromptTemplate.from_messages([
            ("system", get_system_prompt(prompt_path)),
            ("human", get_few_shot_user_prompt(prompt_path))
        ])
        gold_standard_str = ""
        for i, text in enumerate(gold_standard): 
            gold_standard_str += f"Example {i+1}:\n{text}\n\n"
        gold_standard_str = gold_standard_str[:-2]

    # Subsections string
    subsections_str = "\n - " + "\n - ".join(subsections)

    # Adapt excludes
    if excludes != "": 
        excludes = "Excludes: " + excludes

    # Chain
    chain = prompt | llm

    # inputs
    input = {
        "num_samples": num_samples,
        "gold_standard": gold_standard_str,
        "includes": includes,
        "includes_also": includes_also,
        "excludes": excludes,
        "subsections": subsections_str,
        }

    # is topic used? topic in params -> business model must be in user prompt
    if topic is not None:
        if "business_model" not in prompt.input_variables: 
            print("If topic is given, prompt must conatin '{business_model}'")
            return False
        else: 
            if isinstance(topic, list): 
                topic_str = "\n".join([f"{i+1}: {text}" for i, text in enumerate(topic)])
                input["business_model"] = topic_str
            else:
                input["business_model"] = topic

    formatted_prompt = prompt.invoke(input)

    #print("Formatted Prompt:", formatted_prompt)

    # Run
    response = chain.invoke(input)

    #print(response.content)

    return formatted_prompt, response.content

In [10]:
def split_synthetic_data(content: str, num_samples: int): 
    content_list = content.split("\n")
    content_list = [c for c in content_list if c != ""]
    # if len(content_list) != num_samples: 
    #     print("Warning: length of creates examples != num_samples!")
    return content_list

In [11]:
def get_sublevels(nace_class, level): 
    nace_class_temp = nace_class
    nace_id = nace_descriptions[nace_descriptions["CODE"] == nace_class_temp]["ID"].iloc[0]
    nace_class_lvl_2 = []
    nace_class_lvl_3 = []
    nace_class_lvl_4 = []

    for _, row in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id].iterrows(): 
        nace_id_temp = row["ID"]
        nace_class_lvl_2.append(f'{row["NAME"]}')
        for _, row_2 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp].iterrows(): 
            nace_id_temp_temp = row_2["ID"]
            nace_class_lvl_3.append(f'{row["NAME"]}: {row_2["NAME"]}')
            for _, row_3 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp_temp].iterrows(): 
                nace_class_lvl_4.append(f'{row["NAME"]}: {row_2["NAME"]}: {row_3["NAME"]}')
        
    if level == 2: 
        return nace_class_lvl_2
    if level == 3: 
        return nace_class_lvl_3
    if level == 4: 
        return nace_class_lvl_4

In [12]:
def get_sublevels_df(nace_class, level) -> pd.DataFrame: 
    nace_class_temp = nace_class
    nace_id = nace_descriptions[nace_descriptions["CODE"] == nace_class_temp]["ID"].iloc[0]
    nace_class_lvl_2 = []
    nace_class_lvl_3 = []
    nace_class_lvl_4 = []

    for _, row in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id].iterrows(): 
        nace_id_temp = row["ID"]
        nace_class_lvl_2.append(row)
        for _, row_2 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp].iterrows(): 
            nace_id_temp_temp = row_2["ID"]
            nace_class_lvl_3.append(row_2)
            for _, row_3 in nace_descriptions[nace_descriptions["PARENT_ID"] == nace_id_temp_temp].iterrows(): 
                nace_class_lvl_4.append(row_3)
        
    if level == 2: 
        df =  pd.concat(nace_class_lvl_2, axis=1).T
        return df.drop_duplicates()

    if level == 3: 
        df = pd.concat(nace_class_lvl_3, axis=1).T
        return df.drop_duplicates()

    if level == 4: 
        df = pd.concat(nace_class_lvl_4, axis=1).T
        return df.drop_duplicates()

In [13]:
generate_nace_class = "A"

includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
assert includes is not None and includes != ""
includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
includes_also = "" if pd.isna(includes_also) else includes_also
excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
excludes = "" if pd.isna(excludes) else excludes

gold_standard = []

## Generate Few-Shot Data

In [14]:
# select gold standard data

# ds_2_desc  = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

# ds_2_desc  = ds_2_desc[pd.notna(ds_2_desc["Description"])]

# gold_standard = []
# # 1. take one of each lvl 3 class:
# for lvl_3 in ds_2_desc[pd.notna(ds_2_desc["Description"])].groupby("NACE_lvl_3").size().index: 
#     gold_standard.append(ds_2_desc[ds_2_desc["NACE_lvl_3"] == lvl_3].iloc[0])

# df_gold_standard = pd.concat(gold_standard, axis=1).T

#df_gold_standard.to_csv("/Users/hendrikweichel/Downloads/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv")

In [15]:
#df_gold_standard = pd.read_csv("projects/nace_classification/nace_report_topic_analysis/data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")
df_gold_standard = pd.read_csv("data/datasets/reports_subset_from_full_data_2/reports_subset_from_full_data_2_gold_standard_descriptions_for_data_generation.csv", sep=";")

### Hyperparams

In [16]:
prompt_path = "generate_synthetic_data/prompts/prompts_5_list.json"
print("System Prompt: ", get_system_prompt(prompt_path))
print("User Prompt: ", get_few_shot_user_prompt(prompt_path))

System Prompt:  You generate realistic paragraphs from corporate annual reports that are a presentation of the copmanies business model. The text must: - sound natural and specific - avoid textbook or definitional language - vary structure, length, and narrative style - include concrete operational details The text must NOT: - repeat industry definitions - follow a fixed template - explicitly explain what the company does in generic terms
User Prompt:  TASK 
You get list of short descriptions of a companies' business models. Use these descriptions to formulate it into a typical description within an annual report.

DEFINITION
{includes} {includes_also}

{excludes}

Here are some examples of descriptions of these classes: 
```
{gold_standard}
```

BUSINESS MODELS
{business_model}

INSTRUCTIONS
 - For each given Business Model, create a business model description that is in a similar format as the examples
 - Do NOT exactly copy phrases, sentence patterns, or structure from the examples


In [17]:
level = 1
head_nace_code = "1" if level > 1 else None
generated_classes = nace_descriptions[nace_descriptions["PARENT_ID"] == head_nace_code]["CODE"]
generated_classes = ["A", "B", "C", "J", "F"]

In [ ]:
few_shot = True

In [18]:
# generate date 

date = datetime.datetime.now().strftime("%Y%m%d")
suffix = "__few_shot" if few_shot else "__zero_shot"
suffix += "__from_lvl_4_topics"
store_path = "data/synthetic_data/two_step/data_" + date + f"__level_{level}__subclasses_{head_nace_code}__{os.path.basename(prompt_path).replace('.json', '')}{suffix}/" 
os.makedirs(store_path, exist_ok=True)

In [19]:
generated_data = {}

In [21]:
# # load previous results
# prev_results = "projects/nace_classification/nace_report_topic_analysis/data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_5__few_shot/class_A.csv"
# prev_results = "data/synthetic_data/data_20251218__level_1__subclasses_None__prompts_5__few_shot"
# for class_name in generated_classes: 
#    file_path = os.path.join(prev_results, f"class_{class_name}.csv")
#    try:
#        df = pd.read_csv(file_path)
#        generated_data[class_name] = {
#            "data": df[class_name].tolist()
#        }
#    except Exception as e:
#        print(f"Could not load previous results for class {class_name}: {e}")

### Generate Topics

Here: 

- for each class get the subclasses on level 4
- for each subclass: 
    - generate k topics such that n_data topics are created

In [22]:
n_data = 1000 # total number of samples to generate per class
num_samples = 20 # number of samples per iteration (generated through one LLM call)
iterations_ = n_data // num_samples

In [25]:
topics = {}

In [26]:
for generate_nace_class in generated_classes:

    df_subsections = get_sublevels_df(generate_nace_class, level=4)

    topics_per_subsec = math.ceil(n_data / len(df_subsections))
    topic_samples = min(topics_per_subsec, 50)
    topic_iterations = math.ceil(topics_per_subsec / topic_samples)

    print("Nbr. of Subsecs. : ", len(df_subsections), ", Topics per Subclass", topics_per_subsec, ", Samples per LLM call: ", topic_samples, ", LLM calls per Subsec.: ", topic_iterations)

    examples = []

    for i, subsection in tqdm(df_subsections.iterrows()): 

        includes = subsection["Includes"]
        if pd.isna(includes):
            includes = subsection["NAME"]

        includes_also = subsection["IncludesAlso"]
        includes_also = "" if pd.isna(includes_also) else includes_also
        excludes = subsection["Excludes"]
        excludes = "" if pd.isna(excludes) else excludes

        topics_subsection = []

        for i in range(topic_iterations):
            res = generate_topic(num_samples=topic_samples, 
                                includes=includes, 
                                includes_also=includes_also, 
                                excludes=excludes,  
                                prompt_path=prompt_path, 
                                model="gpt-4o-mini", 
                                temperature=0.8)

            topics_subsection.append(res[1])
        
        examples.append({"CODE": subsection["CODE"], "NAME": subsection["NAME"], "TOPICS": "\n".join(topics_subsection)})
    
    clean_text = lambda x: re.sub(r'^\d+\.\s*', " ", x).strip()
    
    examples = pd.DataFrame(examples)
    examples["TOPICS"] = examples["TOPICS"].apply(clean_text)

    data = split_synthetic_data("\n\n".join(examples["TOPICS"]), num_samples * iterations_)

    results = {
        "topics": random.sample(data, k=2),
        "topics_df": examples,
        "system_prompt": res[0].messages[0].content,
        "user_prompt": res[0].messages[1].content,
        "output": data,
        "few_shot": few_shot
    }

    topics[generate_nace_class] = results

Nbr. of Subsecs. :  78 , Topics per Subclass 13 , Samples per LLM call:  13 , LLM calls per Subsec.:  1


78it [09:49,  7.56s/it]


Nbr. of Subsecs. :  33 , Topics per Subclass 31 , Samples per LLM call:  31 , LLM calls per Subsec.:  1


33it [07:23, 13.44s/it]


Nbr. of Subsecs. :  230 , Topics per Subclass 5 , Samples per LLM call:  5 , LLM calls per Subsec.:  1


230it [14:33,  3.80s/it]


Nbr. of Subsecs. :  26 , Topics per Subclass 39 , Samples per LLM call:  39 , LLM calls per Subsec.:  1


26it [07:22, 17.03s/it]


Nbr. of Subsecs. :  22 , Topics per Subclass 46 , Samples per LLM call:  46 , LLM calls per Subsec.:  1


22it [06:54, 18.85s/it]


In [37]:
examples

""


In [38]:
data

['A property development firm specializes in acquiring distressed residential properties, renovating them, and selling them at a premium to first-time homebuyers.',
 '2. A mixed-use development company partners with local governments to transform underutilized land into vibrant communities with both residential and commercial spaces.',
 '3. A real estate investment trust (REIT) focuses on developing luxury condominiums in urban areas, targeting affluent buyers and investors.',
 '4. A sustainable building developer creates eco-friendly residential communities that incorporate renewable energy sources and green building materials.',
 '5. A joint venture between a private developer and a municipal government aims to revitalize downtown areas by building affordable housing and community amenities.',
 '6. A modular construction company designs and manufactures pre-fabricated residential units, reducing construction time and costs for homebuilders.',
 '7. A luxury resort developer constructs

#### Store Topics

In [27]:
all_topics = []
for class_ in topics: 
    df_temp = topics[class_]["topics_df"].copy()
    df_temp["LVL_1_CODE"] = class_
    all_topics.append(df_temp)

all_topics = pd.concat(all_topics, axis=0)
all_topics.to_csv(os.path.join(store_path, "all_topics.csv"))

In [ ]:
topics_store = copy.deepcopy(topics)
for class_ in topics_store: 
    topics_store[class_]["topics_df"] = None

with open(os.path.join(store_path, "topics.json"), "w") as f: 
    json.dump(topics_store, f, indent=4)

### Generate Descriptions

In [29]:
generated_data = {}

In [ ]:
for generate_nace_class in generated_classes:

    includes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Includes"].item()
    if pd.isna(includes):
        print("No description for class:", generate_nace_class)
        continue
    includes_also = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["IncludesAlso"].item()
    includes_also = "" if pd.isna(includes_also) else includes_also
    excludes = nace_descriptions[nace_descriptions["CODE"] == generate_nace_class]["Excludes"].item()
    excludes = "" if pd.isna(excludes) else excludes

    if few_shot: 
        gold_standard = df_gold_standard[df_gold_standard["NACE_letter"] == generate_nace_class]["Description_clean"].to_list()[:3] 
        gold_standard = [text.replace("\n", "") for text in gold_standard]
    else: 
        gold_standard = [] 

    subsections = get_sublevels(generate_nace_class, level=2)

    examples = []

    #if generated_data.get(generate_nace_class) is not None:
    #    if len(generated_data[generate_nace_class].get("data", [])) > 0:
    #        examples = generated_data[generate_nace_class]["data"]

    for i in tqdm(range(iterations_), desc=generate_nace_class):

        iteration_topics = topics[generate_nace_class]["output"][num_samples*i:num_samples*(i+1)]
        res = generate_synthetic_data(num_samples=num_samples, gold_standard=gold_standard, includes=includes, includes_also=includes_also, excludes=excludes, subsections=subsections, prompt_path=prompt_path, model="gpt-4o-mini", topic=iteration_topics)
        examples.append(res[1])
        data = split_synthetic_data("\n\n".join(examples), num_samples * iterations_)
        pd.DataFrame(data, columns=[generate_nace_class]).to_csv(os.path.join(store_path, f"class_{generate_nace_class}.csv"), index=False)

        if i < 2: 
            [print(example) for example in examples]
            print(res[0].messages[1].content)

        if len(data) >= num_samples * iterations_:
            break

    results = {
        "data": data,
        "prompt": res[0],
        "system_prompt": res[0].messages[0].content,
        "user_prompt": res[0].messages[1].content,
        "output": examples, 
        "few_shot": few_shot
    }

    generated_data[generate_nace_class] = results

A:   0%|                                                 | 0/50 [00:00<?, ?it/s]

A:   2%|▊                                        | 1/50 [00:27<22:35, 27.66s/it]

1. Operating as a cooperative, our organization brings together local farmers to harness their collective strengths in cultivating high-quality organic wheat and barley. By pooling resources, we not only enhance our production capabilities but also streamline the marketing process, allowing us to supply specialty bakeries and restaurants with premium grains. This collaborative approach fosters a strong sense of community while ensuring that our products meet the rigorous standards of discerning culinary professionals.

2. Our subscription service revolutionizes the way consumers access fresh, seasonal legumes, including chickpeas and lentils. Through a user-friendly online platform, we connect directly with customers, emphasizing transparency from farm to table. Each delivery showcases the best of what our farms have to offer, allowing families to enjoy nutritious meals while supporting sustainable agricultural practices. This model not only caters to health-conscious consumers but als

A:   4%|█▋                                       | 2/50 [00:55<22:23, 27.98s/it]

1. Operating as a cooperative, our organization brings together local farmers to harness their collective strengths in cultivating high-quality organic wheat and barley. By pooling resources, we not only enhance our production capabilities but also streamline the marketing process, allowing us to supply specialty bakeries and restaurants with premium grains. This collaborative approach fosters a strong sense of community while ensuring that our products meet the rigorous standards of discerning culinary professionals.

2. Our subscription service revolutionizes the way consumers access fresh, seasonal legumes, including chickpeas and lentils. Through a user-friendly online platform, we connect directly with customers, emphasizing transparency from farm to table. Each delivery showcases the best of what our farms have to offer, allowing families to enjoy nutritious meals while supporting sustainable agricultural practices. This model not only caters to health-conscious consumers but als

A:  28%|███████████▏                            | 14/50 [06:50<17:59, 30.00s/it]

In [40]:
topics[generate_nace_class]["output"]

['A cooperative model where local farmers pool resources to collectively grow and market high-quality organic wheat and barley to specialty bakeries and restaurants.',
 '2. A subscription-based service offering fresh, seasonal legumes such as chickpeas and lentils directly to consumers through an online platform, promoting farm-to-table transparency.',
 '3. A vertically integrated agribusiness that cultivates sunflowers and processes them into premium cooking oil, distributing through both retail and food service channels.',
 '4. A regenerative agriculture initiative focusing on the cultivation of diverse cereal crops, paired with educational programs for sustainable farming practices targeted at new farmers.',
 '5. A niche market business that specializes in the cultivation and export of rare leguminous crops like lupines and pigeon peas, catering to health food stores worldwide.',
 '6. An agri-tech startup that leverages data analytics and IoT devices to optimize the growing conditio

In [33]:
topics[generate_nace_class]["topics_df"]#[num_samples*i:num_samples*(i+1)]

In [ ]:
# print prompts
for k,v in generated_data.items(): 
    print(k,len(v["output"]), "_______"*20)
    for k in v["output"]: 
        print(k)
        print("_")

C 25 ____________________________________________________________________________________________________________________________________________
1. Our company is at the forefront of innovation in the recreational vehicle sector, specializing in the design and production of collapsible trailers that redefine convenience for outdoor enthusiasts. These trailers are engineered for easy storage and transport, allowing users to maximize their adventures without the hassle of cumbersome equipment. By focusing on lightweight materials and smart engineering, we ensure that our products not only meet the demands of the modern adventurer but also enhance the overall experience of outdoor travel. Our commitment to quality and functionality has positioned us as a preferred choice among recreational vehicle users who value both practicality and design.

2. As a dedicated manufacturer of specialized equipment tailored for off-road vehicles, we cater to the growing adventure tourism market. Our prod

In [ ]:
# print prompts
for k,v in generated_data.items(): 
    print(k,"_______"*20)
    print(v["user_prompt"])


A ____________________________________________________________________________________________________________________________________________

TASK 
You get a short description of a companies' busines model and rephrase it such that it fits into a typical description within an annual report.

DEFINITION
This section includes the exploitation of vegetal and animal natural resources, comprising the activities of growing of crops, raising and breeding of animals, harvesting of timber and other plants, animals or animal products from a farm or their natural habitats. 



Here are some examples of descriptions of these classes: 
```
Example 1:
The company, together with its subsidiaries, operates as one of the largest vertically integrated agricultural groups in Ukraine, engaging in the production, storage, processing, and sale of agricultural products. The company's key activities include breeding pigs, processing pork, and producing wheat and sunflower. The Group focuses on three winter 

#### Aggregate data and split

In [ ]:
# config

config = {
    "prompts": {k: v.get("user_prompt") for k, v in generated_data.items()}, 
    #"samples": num_samples * iterations_,
    #"generated_iterations": iterations_,
    "level": level,
    "head_nace_code": head_nace_code,
    "system_prompt": get_system_prompt(prompt_path), 
    "few_shot_prompting": few_shot
}

# store
import json
with open(os.path.join(store_path, "config.json"), "w") as f: 
    json.dump(config, f, indent=4)

In [ ]:
df_full = []
for k, v in generated_data.items(): 
    df_temp = pd.DataFrame(v["data"], columns=["text"])
    df_temp["label"] = k
    df_full.append(df_temp)
df_full = pd.concat(df_full, axis=0)
df_full = df_full.reset_index(drop=True)
df_full

,text,label
0,1. Our company is at the forefront of innovati...,C
1,2. As a dedicated manufacturer of specialized ...,C
2,3. At the heart of our woodworking company is ...,C
3,4. We are revolutionizing the logistics indust...,C
4,5. Our company provides mobile conveyor system...,C
...,...,...
995,16. We offer a fishing gear rental service tha...,A
996,17. Our woodworking school teaches students th...,A
997,18. Operating as a regenerative agriculture fa...,A
998,19. Our fishing tourism venture offers guided ...,A


In [ ]:
import re
clean_text = lambda x: re.sub(r'^\d+\.\s*', " ", x).strip()

In [ ]:
df_full["text"] = df_full["text"].apply(clean_text)

In [ ]:
df_full.to_csv(os.path.join(store_path, "synthetic_data_full.csv"), index=False)

In [ ]:
# make train test split 6:2:2

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df_full, test_size=0.4, random_state=42, stratify=df_full["label"])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

len(train_df), len(test_df), len(val_df)

(600, 200, 200)

In [ ]:
train_df.to_csv(os.path.join(store_path, "train_data.csv"), index=False)
test_df.to_csv(os.path.join(store_path, "test_data.csv"), index=False)
val_df.to_csv(os.path.join(store_path, "val_data.csv"), index=False)

In [272]:
llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0.8
        )

In [ ]:
a =llm.invoke("""TASK 
You get list of short descriptions of a companies' business models. Use these descriptions to formulate it into a typical description within an annual report.

DEFINITION
This section includes the exploitation of vegetal and animal natural resources, comprising the activities of growing of crops, raising and breeding of animals, harvesting of timber and other plants, animals or animal products from a farm or their natural habitats. 



Here are some examples of descriptions of these classes: 
```
Example 1:
The company, together with its subsidiaries, operates as one of the largest vertically integrated agricultural groups in Ukraine, engaging in the production, storage, processing, and sale of agricultural products. The company's key activities include breeding pigs, processing pork, and producing wheat and sunflower. The Group focuses on three winter crops and two summer crops, maintaining a crop ratio of 60% winter crops (wheat, barley, rapeseed) and 40% summer crops (sunflower, corn). This strategy is designed to ensure a steady supply of basic food products, which remain in high demand, especially during wartime. The Group has also secured additional financing to cover key production costs and maintain strategic reserves of essential supplies.

Example 2:
The company initially focused on the business activities of catching skipjack and red snapper with a target of export sales. Over time, in 1983, the Company started its first production operations which was marked by the establishment of its factory in Kendari, Southeast Sulawesi.Furthermore, in order to expand its market share, the Company expanded into the integrated fish processing industry which includes processing activities. Since then, the Company has been able to produce processed marine products that contain high protein and added value, such as fish filets, tuna, octopus, squid, and other value-added products.In managing its business, the Company is committed to always consider sustainability values in all aspects. Not merely focusing on financial performance, the Company also promotes business alignment and harmonization as well as provides optimal benefits to stakeholders. The Company believes that by establishing a mutually beneficial business ecosystem, long-term business continuity will be maintained.

Example 3:
The company is the global leader in the full cycle breeding, production and sale of Yellowtail Kingfish and is renowned world-wide for its exceptionally high quality fish. Our company is recognised for innovation and its high degree of expertise in the farming of Yellowtail Kingfish. We are the largest producer of aquaculture Yellowtail Kingfish outside of Japan. Our diverse customer base has long appreciated the consistently high quality of our fish and our reliability in supplying our fresh and frozen range to markets all over the world 52 weeks of the year.
```

BUSINESS MODELS
0: A tech platform connects local wood suppliers with artisans looking for raw materials, streamlining the procurement process.
1: A company that offers consulting services to help landowners manage their forests for both timber and non-timber product extraction.
2: An organic dairy farm producing artisanal cheeses and yogurts, paired with a farm-to-table café on-site.
3: A cooperative that aggregates products from various mixed farms to sell under a single organic brand.
4: A sustainable sea cucumber farm that caters to niche markets in Asian health and culinary sectors.
5: An eco-friendly farm that integrates permaculture principles, offering educational courses and a variety of organic seasonal produce.

INSTRUCTIONS
 - For each given Business Model, create a business model description that is in a similar format as the examples
 - Do NOT exactly copy phrases, sentence patterns, or structure from the examples
 - Think of the examples as constraints, not templates
 - Also use terms other than those used in the sector definition to make the examples more diverse""")

In [279]:
print(a.content)

### Business Model Descriptions

**Business Model 0:**
The company operates an innovative tech platform that seamlessly connects local wood suppliers with artisans seeking high-quality raw materials for their crafts. By streamlining the procurement process, the platform enhances efficiency and accessibility for both suppliers and buyers. In addition to facilitating transactions, the company emphasizes transparency and sustainability, ensuring that sourced materials are harvested responsibly. This approach not only supports local economies but also fosters a vibrant community of creators dedicated to sustainable practices in their production processes.

**Business Model 1:**
Our company specializes in providing expert consulting services tailored for landowners aiming to optimize the management of their forests. We focus on both timber and non-timber product extraction, guiding clients through sustainable practices that enhance productivity while preserving ecosystem integrity. Our team

In [280]:
b = llm.invoke("""TASK 
You get list of short descriptions of a companies' business models. Use these descriptions to formulate it into a typical description within an annual report.

DEFINITION
This section includes the exploitation of vegetal and animal natural resources, comprising the activities of growing of crops, raising and breeding of animals, harvesting of timber and other plants, animals or animal products from a farm or their natural habitats. 



Here are some examples of descriptions of these classes: 
```
Example 1:
The company, together with its subsidiaries, operates as one of the largest vertically integrated agricultural groups in Ukraine, engaging in the production, storage, processing, and sale of agricultural products. The company's key activities include breeding pigs, processing pork, and producing wheat and sunflower. The Group focuses on three winter crops and two summer crops, maintaining a crop ratio of 60% winter crops (wheat, barley, rapeseed) and 40% summer crops (sunflower, corn). This strategy is designed to ensure a steady supply of basic food products, which remain in high demand, especially during wartime. The Group has also secured additional financing to cover key production costs and maintain strategic reserves of essential supplies.

Example 2:
The company initially focused on the business activities of catching skipjack and red snapper with a target of export sales. Over time, in 1983, the Company started its first production operations which was marked by the establishment of its factory in Kendari, Southeast Sulawesi.Furthermore, in order to expand its market share, the Company expanded into the integrated fish processing industry which includes processing activities. Since then, the Company has been able to produce processed marine products that contain high protein and added value, such as fish filets, tuna, octopus, squid, and other value-added products.In managing its business, the Company is committed to always consider sustainability values in all aspects. Not merely focusing on financial performance, the Company also promotes business alignment and harmonization as well as provides optimal benefits to stakeholders. The Company believes that by establishing a mutually beneficial business ecosystem, long-term business continuity will be maintained.

Example 3:
The company is the global leader in the full cycle breeding, production and sale of Yellowtail Kingfish and is renowned world-wide for its exceptionally high quality fish. Our company is recognised for innovation and its high degree of expertise in the farming of Yellowtail Kingfish. We are the largest producer of aquaculture Yellowtail Kingfish outside of Japan. Our diverse customer base has long appreciated the consistently high quality of our fish and our reliability in supplying our fresh and frozen range to markets all over the world 52 weeks of the year.
```

BUSINESS MODELS
0: A tech platform connects local wood suppliers with artisans looking for raw materials, streamlining the procurement process.
1: A company that offers consulting services to help landowners manage their forests for both timber and non-timber product extraction.
2: An organic dairy farm producing artisanal cheeses and yogurts, paired with a farm-to-table café on-site.
3: A cooperative that aggregates products from various mixed farms to sell under a single organic brand.
4: A sustainable sea cucumber farm that caters to niche markets in Asian health and culinary sectors.
5: An eco-friendly farm that integrates permaculture principles, offering educational courses and a variety of organic seasonal produce.
6: A mixed operation that incorporates agro-tourism, allowing visitors to experience crop harvesting while selling farm products on-site.
7: A forestry management consultancy that assists landowners in optimizing timber yield while maintaining ecological health in their forests.
8: A boutique retail shop that curates and sells handcrafted goods made from roundwood and wild forest products.
9: An organic vineyard that not only produces wine but also operates a farm-to-glass restaurant, providing farm tours and tastings.
10: A timber construction company specializing in homes built from locally sourced roundwood, emphasizing energy efficiency.
11: A community-supported fishery (CSF) that offers subscription boxes of seasonal, wild-caught fish directly from local fishermen to households.
12: A multi-generational family business that has been producing firewood and charcoal using traditional methods for decades.
13: A farm specializing in heirloom vegetables, providing seed shares to local gardeners and running seasonal pop-up markets.
14: A mixed farming operation that grows vegetables alongside specialty mushrooms, supplying local chefs with unique ingredients.
15: An online marketplace for niche forest products, connecting small producers of non-timber goods with a broader audience.
16: A permaculture farm that integrates tree crops, vegetables, and livestock to create a self-sustaining ecosystem.
17: An agribusiness that raises goats for milk and cheese production, selling to local markets and hosting farm visits for educational purposes.
18: A specialty farm that produces organic spices and herbs, targeting gourmet chefs and health food stores with unique offerings.
19: A cooperative that brings together mushroom growers and crop farmers to enhance biodiversity and offer unique farm products.

INSTRUCTIONS
 - For each given Business Model, create a business model description that is in a similar format as the examples
 - Do NOT exactly copy phrases, sentence patterns, or structure from the examples
 - Think of the examples as constraints, not templates
 - Also use terms other than those used in the sector definition to make the examples more diverse""")

In [281]:
print(b.content)

### Business Model Descriptions

**Business Model 0:**  
The company operates a cutting-edge digital platform that serves as a bridge between local timber suppliers and artisans in need of raw materials. By streamlining the procurement process, the platform enhances efficiency and fosters direct connections among stakeholders, ensuring that artisans can access quality wood sustainably sourced from nearby regions. This innovative approach not only supports local economies but promotes responsible forestry practices.

**Business Model 1:**  
This organization specializes in providing expert consulting services to landowners, focusing on sustainable forest management practices. The firm aids clients in maximizing both timber and non-timber product yields, while prioritizing ecological preservation. Through tailored management plans and ongoing support, the company empowers landowners to achieve economic benefits alongside environmental stewardship.

**Business Model 2:**  
Located on a pi